# Capstone Project — AI-Powered Intelligent Search System

**Dataset:** News Category Dataset (Kaggle)  
**Embeddings:** Sentence Transformers (`all-MiniLM-L6-v2`)  
**Vector Databases Compared:** FAISS vs. ChromaDB

## Step 1 — Install Dependencies

In [ ]:
!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install chromadb
!pip -q install pandas
!pip -q install numpy

In [ ]:
import pandas as pd
import numpy as np
import time
import re

from sentence_transformers import SentenceTransformer
import faiss
import chromadb

## Step 2 — Upload the Dataset

In [1]:
from google.colab import files

uploaded_files = files.upload()

Saving archive.zip to archive.zip


In [1]:
import zipfile
import os

uploaded_name = list(uploaded_files.keys())[0]

if uploaded_name.endswith(".zip"):
    with zipfile.ZipFile(uploaded_name, "r") as zip_ref:
        zip_ref.extractall(".")
    print("Zip file extracted.")

print("Files available now:")
print(os.listdir("."))

Zip file extracted.
Files available now:
['.config', 'archive.zip', 'News_Category_Dataset_v3.json', 'sample_data']


## Step 3 — Load the Dataset

In [1]:
dataset_path = "News_Category_Dataset_v3.json"

news_df = pd.read_json(dataset_path, lines=True)

print("Shape of dataset:", news_df.shape)
news_df.head()

Shape of dataset: (209527, 6)


## Step 4 — Data Preprocessing and Cleaning

In [1]:
news_df = news_df[["category", "headline", "short_description", "authors", "date", "link"]]

news_df = news_df.dropna(subset=["headline", "short_description"])

news_df["text"] = news_df["headline"] + ". " + news_df["short_description"]

news_df = news_df[news_df["text"].str.strip() != ""]

print("Shape after removing empty rows:", news_df.shape)

Shape after removing empty rows: (209527, 7)


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)            # strip URLs
    text = re.sub(r"[^a-z0-9\s.,!?']", " ", text)   # strip special characters
    text = re.sub(r"\s+", " ", text).strip()         # collapse whitespace
    return text


news_df["clean_text"] = news_df["text"].apply(clean_text)

news_df[["text", "clean_text"]].head()

In [1]:
SAMPLE_SIZE = 5000

sample_df = news_df.sample(n=SAMPLE_SIZE, random_state=42)
sample_df = sample_df.reset_index(drop=True)

sample_df["doc_id"] = sample_df.index.astype(str)

print("Final dataset shape used for the project:", sample_df.shape)
sample_df.head()

Final dataset shape used for the project: (5000, 9)


## Step 5 — Generate Embeddings with Sentence Transformers

`all-MiniLM-L6-v2` is small, fast, and performs well on semantic search tasks.

In [1]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

In [1]:
embed_start = time.time()

doc_embeddings = embed_model.encode(
    sample_df["clean_text"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True,
)

embed_end = time.time()
embedding_time = embed_end - embed_start

print("Embedding Shape:", doc_embeddings.shape)
print("Time taken to generate embeddings:", round(embedding_time, 2), "seconds")

Embedding Shape: (5000, 384)
Time taken to generate embeddings: 128.11 seconds


## Step 6 — Store Embeddings in FAISS

FAISS expects `float32` vectors; `IndexFlatL2` measures similarity via Euclidean (L2) distance.

In [1]:
faiss_vectors = doc_embeddings.astype("float32")

vec_dim = faiss_vectors.shape[1]
faiss_index = faiss.IndexFlatL2(vec_dim)

faiss_index_start = time.time()
faiss_index.add(faiss_vectors)
faiss_index_end = time.time()
faiss_indexing_time = faiss_index_end - faiss_index_start

print("Total vectors stored in FAISS:", faiss_index.ntotal)
print("FAISS indexing time:", round(faiss_indexing_time, 4), "seconds")

Total vectors stored in FAISS: 5000
FAISS indexing time: 0.0215 seconds


## Step 7 — Store Embeddings in ChromaDB

ChromaDB stores embeddings alongside metadata (category, headline, etc.) so search results can be displayed with rich context.

In [ ]:
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("news_collection")
except Exception:
    pass

news_collection = chroma_client.create_collection(name="news_collection")

In [1]:
doc_metadata = [
    {
        "category": row["category"],
        "headline": row["headline"],
        "short_description": row["short_description"],
    }
    for _, row in sample_df.iterrows()
]

chroma_index_start = time.time()

news_collection.add(
    ids=sample_df["doc_id"].tolist(),
    embeddings=doc_embeddings.tolist(),
    metadatas=doc_metadata,
)

chroma_index_end = time.time()
chroma_indexing_time = chroma_index_end - chroma_index_start

print("Total Records Stored in ChromaDB:", news_collection.count())
print("ChromaDB indexing time:", round(chroma_indexing_time, 4), "seconds")

Total Records Stored in ChromaDB: 5000
ChromaDB indexing time: 2.9319 seconds


## Step 8 — Natural Language Search Functions

One function per backend keeps the search logic from being duplicated for every new query.

In [ ]:
def search_faiss(query, top_k=5):
    q_vec = embed_model.encode([query], convert_to_numpy=True).astype("float32")

    t0 = time.time()
    distances, indices = faiss_index.search(q_vec, top_k)
    elapsed = time.time() - t0

    print(f"\nFAISS Search Results for Query: '{query}'")
    print(f"Query Time: {round(elapsed, 4)} seconds")

    for rank, idx in enumerate(indices[0]):
        row = sample_df.iloc[idx]
        print(f"\nRank: {rank + 1}")
        print("Category:", row["category"])
        print("Headline:", row["headline"])
        print("Description:", row["short_description"])
        print("Distance Score:", round(distances[0][rank], 4))

    return elapsed

In [ ]:
def search_chroma(query, top_k=5):
    q_vec = embed_model.encode(query, convert_to_numpy=True)

    t0 = time.time()
    results = news_collection.query(query_embeddings=[q_vec.tolist()], n_results=top_k)
    elapsed = time.time() - t0

    print(f"\nChromaDB Search Results for Query: '{query}'")
    print(f"Query Time: {round(elapsed, 4)} seconds")

    for rank in range(len(results["ids"][0])):
        meta = results["metadatas"][0][rank]
        print(f"\nRank: {rank + 1}")
        print("Category:", meta["category"])
        print("Headline:", meta["headline"])
        print("Description:", meta["short_description"])
        print("Distance Score:", round(results["distances"][0][rank], 4))

    return elapsed

## Step 9 — Try a Natural Language Query

Example queries to try: *"new technology in smartphones"*, *"sports championship win"*, *"political election results"*.

In [1]:
user_query = input("Enter Your Search Query: ")

faiss_query_time = search_faiss(user_query, top_k=5)

Enter Your Search Query: new technology in smartphones

FAISS Search Results for Query: 'new technology in smartphones'
Query Time: 0.0354 seconds

Rank: 1
Category: TECH
Headline: Why Not Everyone Wants The Latest iPhone
Description: Remember flip phones? They're still here.
Distance Score: 1.1506

Rank: 2
Category: WELLNESS
Headline: 5 Ways to Turn Your Mobile Device Into a Mecca for Healing
Description: As someone who has spent the past two decades in search of recovery and healing, I am a believer in the power of technology to help us make good transitions to facilitate well-being.
Distance Score: 1.1817

Rank: 3
Category: TECH
Headline: Google Launches New Program To Speed Up Mobile Web
Description: Google unveiled a new open source initiative today that it's calling the Accelerated Mobile Pages Project. The name kind
Distance Score: 1.1863

Rank: 4
Category: PARENTING
Headline: The Touch-Screen Generation
Description: On a chilly day last spring, a few dozen developers of childre

In [1]:
chroma_query_time = search_chroma(user_query, top_k=5)

ChromaDB Search Results for Query: 'new technology in smartphones'
Query Time: 0.0036 seconds

(same ranked results as the FAISS search above, retrieved from the same embeddings)


## Step 10 — Compare Both Databases on Multiple Queries

Run a batch of sample queries against both backends and record the query time for each.

In [1]:
benchmark_queries = [
    "new technology gadgets",
    "sports championship win",
    "political election news",
    "celebrity entertainment news",
    "health and fitness tips",
]

faiss_query_times = []
chroma_query_times = []

for q in benchmark_queries:
    f_time = search_faiss(q, top_k=3)
    c_time = search_chroma(q, top_k=3)

    faiss_query_times.append(f_time)
    chroma_query_times.append(c_time)

FAISS Search Results for Query: 'new technology gadgets'
Query Time: 0.0011 seconds
(top matches: home gadgets/appliances, a 'Bacon Bowl' gadget piece, a Samsung sleep-tracking device story)

ChromaDB Search Results for Query: 'new technology gadgets'
Query Time: 0.0029 seconds
(same top matches as FAISS)

FAISS Search Results for Query: 'sports championship win'
Query Time: 0.0010 seconds
(top matches: a LeBron James piece, a Dean Smith legacy piece, an NCAA bracket joke)

ChromaDB Search Results for Query: 'sports championship win'
Query Time: 0.0027 seconds
(same top matches as FAISS)

FAISS Search Results for Query: 'political election news'
Query Time: 0.0012 seconds
(top matches: a Scaramucci cable-news piece, a HuffPost morning newsbrief, an election/Facebook piece)

ChromaDB Search Results for Query: 'political election news'
Query Time: 0.0027 seconds
(same top matches as FAISS)

FAISS Search Results for Query: 'celebrity entertainment news'
Query Time: 0.0010 seconds
(top mat

## Step 11 — Performance Comparison Table

In [1]:
performance_table = pd.DataFrame({
    "Query": benchmark_queries,
    "FAISS Query Time (s)": [round(t, 5) for t in faiss_query_times],
    "ChromaDB Query Time (s)": [round(t, 5) for t in chroma_query_times],
})

performance_table

                          Query  FAISS Query Time (s)  ChromaDB Query Time (s)
0       new technology gadgets                0.00105                   0.00292
1      sports championship win                0.00100                   0.00269
2      political election news                0.00117                   0.00268
3  celebrity entertainment news                0.00101                   0.00274
4       health and fitness tips                0.00098                   0.00242


In [1]:
print("Indexing Time Comparison")
print("-" * 40)
print("FAISS Indexing Time   :", round(faiss_indexing_time, 4), "seconds")
print("ChromaDB Indexing Time:", round(chroma_indexing_time, 4), "seconds")

print("\nAverage Query Time Comparison")
print("-" * 40)
print("FAISS Average Query Time   :", round(np.mean(faiss_query_times), 5), "seconds")
print("ChromaDB Average Query Time:", round(np.mean(chroma_query_times), 5), "seconds")

Indexing Time Comparison
----------------------------------------
FAISS Indexing Time   : 0.0215 seconds
ChromaDB Indexing Time: 2.9319 seconds

Average Query Time Comparison
----------------------------------------
FAISS Average Query Time   : 0.00104 seconds
ChromaDB Average Query Time: 0.00269 seconds


## Step 12 — Comparison and Justification: FAISS vs. ChromaDB

**FAISS**
- Very fast for indexing and searching raw vectors.
- Doesn't store metadata on its own — that's handled separately here via the `sample_df` DataFrame.
- Best suited when raw search speed matters most and metadata management can be handled independently.

**ChromaDB**
- Somewhat slower than FAISS, but keeps embeddings and metadata together in one place.
- Easier for beginners since there's no separate DataFrame lookup needed for results.
- Best suited for a simple, all-in-one solution with metadata built in.

**Conclusion for this project (News Category Dataset):** Since this dataset carries rich metadata (category, headline, description) that's meant to be shown with every result, ChromaDB is the more convenient option because it keeps everything bundled together. That said, at a much larger scale — millions of records where raw speed matters most — FAISS would be the stronger choice, since it's lighter and faster, with metadata handled separately.

For this ~5,000-record capstone, both databases perform well: ChromaDB is recommended for ease of use, and FAISS is recommended when raw speed is the priority.